# Indian Crop Production and Yield Analysis (1997 - 2015)
## Comprehensive Data Analytics, Gujarat Case Study, and Statistical Insight Report

**Author:** Senior Data Analyst & Software Engineer  
**Dataset:** Kaggle Indian Crop Production Dataset (246,091 records)  
**Focus:** End-to-End Data Cleaning, Exploratory Data Analysis, Derived Metrics, Statistical Modeling, and Gujarat State Deep-Dive


---  
## 1. Environment Setup & Data Loading
We begin by importing core data analysis libraries (`pandas`, `numpy`, `matplotlib`, `seaborn`, `scipy`) and loading the Kaggle crop production dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set aesthetic parameters
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

# Load Dataset
df = pd.read_csv('../data/crop_production.csv')
print(f"Raw Dataset Shape: {df.shape}")
display(df.head())
df.info()

---  
## 2. Phase 1 & 2 — Data Cleaning & Standardization
- Standardize column names to lowercase snake_case.
- Strip whitespace from string columns (e.g. `'Coconut '` -> `'Coconut'`).
- Check for duplicate records.
- Handle missing production values without imposing artificial zeroes.

In [ ]:
# 1. Standardize column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

# 2. Clean string columns
for col in ['state_name', 'district_name', 'season', 'crop']:
    df[col] = df[col].astype(str).str.strip()

# 3. Check for duplicates
dup_count = df.duplicated().sum()
print(f"Duplicates found: {dup_count}")
if dup_count > 0:
    df = df.drop_duplicates()

# 4. Check missing values
print("\nMissing values count:")
print(df.isnull().sum())

# Convert numeric
df['area'] = pd.to_numeric(df['area'], errors='coerce')
df['production'] = pd.to_numeric(df['production'], errors='coerce')

---  
## 3. Phase 3 & 4 — Data Validation & Derived Metrics (Crop Yield)
We calculate **Yield** in Tonnes per Hectare ($Yield = \frac{Production}{Area}$) and **Yield in Kg/Ha** ($Yield_{kg/ha} = Yield \times 1000$).

In [ ]:
# Calculate derived yield
df['yield'] = np.where(
    (df['area'] > 0) & (df['production'].notna()),
    df['production'] / df['area'],
    np.nan
)
df['yield_kg_per_ha'] = df['yield'] * 1000

print(f"Year Range: {df['crop_year'].min()} to {df['crop_year'].max()}")
print(f"Unique States: {df['state_name'].nunique()}")
print(f"Unique Crops: {df['crop'].nunique()}")
df[['area', 'production', 'yield']].describe()

---  
## 4. Phase 5 — Indian National Crop Production & Cultivated Area Trends

In [ ]:
yearly_stats = df.groupby('crop_year').agg(
    total_prod=('production', lambda x: x.sum() / 1e6),
    total_area=('area', lambda x: x.sum() / 1e6)
).reset_index()

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(yearly_stats['crop_year'], yearly_stats['total_prod'], marker='o', color='#1f77b4', linewidth=2, label='Production (M Tonnes)')
ax1.set_xlabel('Crop Year')
ax1.set_ylabel('Production (Million Tonnes)', color='#1f77b4')
ax1.tick_params(axis='y', labelcolor='#1f77b4')

ax2 = ax1.twinx()
ax2.plot(yearly_stats['crop_year'], yearly_stats['total_area'], marker='s', color='#2ca02c', linewidth=2, linestyle='--', label='Area (M Hectares)')
ax2.set_ylabel('Cultivated Area (Million Hectares)', color='#2ca02c')
ax2.tick_params(axis='y', labelcolor='#2ca02c')

plt.title('India: Crop Production and Cultivated Area Over Time (1997 - 2015)', fontweight='bold', fontsize=14)
plt.show()

---  
## 5. Phase 6 — Gujarat State Case Study
Gujarat is analyzed specifically to determine its agricultural breakdown, top cash crops (Groundnut, Cotton, Sugarcane), and productivity.

In [ ]:
gujarat = df[df['state_name'].str.lower() == 'gujarat'].copy()
print(f"Gujarat Dataset Rows: {len(gujarat)}")
print(f"Gujarat Districts: {gujarat['district_name'].nunique()}")

guj_top_crops = gujarat.groupby('crop')['production'].sum().sort_values(ascending=False).head(10) / 1e6

plt.figure(figsize=(10, 5))
sns.barplot(x=guj_top_crops.values, y=guj_top_crops.index, hue=guj_top_crops.index, palette='rocket', legend=False)
plt.title('Gujarat: Top 10 Crops by Production Volume (Million Tonnes)', fontweight='bold')
plt.xlabel('Production (Million Tonnes)')
plt.show()

---  
## 6. Phase 7 — Statistical Analysis & Hypothesis Testing (ANOVA)
We test whether crop yield significantly differs across seasons using One-Way ANOVA.

In [ ]:
# Pearson Correlation between Area and Production
corr, p_val = stats.pearsonr(df['area'].dropna(), df['production'].dropna())
print(f"Pearson Correlation (Area vs Production): r = {corr:.4f} (p-value: {p_val})")

# One-Way ANOVA for Yield across Seasons
season_groups = [group['yield'].dropna() for _, group in df.groupby('season') if len(group['yield'].dropna()) > 30]
f_stat, p_val_anova = stats.f_oneway(*season_groups)
print(f"ANOVA F-Statistic: {f_stat:.4f}, p-value: {p_val_anova:.4e}")
if p_val_anova < 0.05:
    print("Conclusion: Statistically significant difference in crop yields across different seasons (p < 0.05).")

---  
## 7. Conclusions & Strategic Policy Recommendations
1. **High Crop Volume Concentration:** Coconut, Sugarcane, and Rice constitute over 85% of total recorded production weight.
2. **Gujarat's Specialization:** Gujarat excels in cash crops including Groundnut, Cotton, and Sugarcane, displaying strong productivity.
3. **Seasonal Variation:** Kharif and Rabi seasons remain the dominant agricultural drivers in India.